In [ ]:
# ============================================================
# 08_Robustness.ipynb
# Section 1 — Setup, Data, Rebuilt Machinery
# Purpose: Frozen data + metrics + LSTM sequence builders (NB05)
#          + Chronos embedding pipeline (NB06). Extract and SAVE
#          ctx=100 embeddings for reuse.
# ============================================================
!pip install -q chronos-forecasting


import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_DIR   = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR   = os.path.join(PROJECT_ROOT, "figures")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001
N_FEATURES = len(FINAL_FEATURES_FD001)

print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "STOP — enable the T4 GPU first!"
DEVICE = "cuda"

# --- Data ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
y_true = test_last["RUL_true"].values
units_test = test_last["unit"].values
assert train.shape[0] == 20631 and test_last.shape[0] == 100
print("✓ Data loaded.")

# --- Metrics (identical to NB03-07) ---
def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0) - 1.0,
                                         np.exp(d/10.0) - 1.0)))

# --- LSTM sequence builders (identical to NB05) ---
def build_train_sequences(df, window, label_col):
    X_list, y_list = [], []
    for unit, grp in df.groupby("unit"):
        grp = grp.sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        labels = grp[label_col].values
        for end in range(len(grp)):
            start = max(0, end - window + 1)
            seq = feats[start:end + 1]
            if len(seq) < window:
                seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
            X_list.append(seq); y_list.append(labels[end])
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list, dtype=np.float32))

def build_last_window_sequences(df_full, window, units_order):
    X_list = []
    for unit in units_order:
        grp = df_full[df_full["unit"] == unit].sort_values("cycle")
        feats = grp[FINAL_FEATURES_FD001].values
        seq = feats[-window:]
        if len(seq) < window:
            seq = np.vstack([np.zeros((window - len(seq), N_FEATURES)), seq])
        X_list.append(seq)
    return np.array(X_list, dtype=np.float32)

# --- Chronos-2 pipeline + embedding extraction (identical to NB06) ---
from chronos import Chronos2Pipeline
pipe2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=DEVICE)
print("✓ Chronos-2 loaded.")

def pool_embedding(emb):
    e = emb.detach().float().cpu().numpy()
    e = np.squeeze(e)
    if e.ndim == 1: return e
    return e.reshape(-1, e.shape[-1]).mean(axis=0)

def extract_embeddings(windows, batch_size=64, desc=""):
    x = torch.tensor(windows.transpose(0, 2, 1), dtype=torch.float32)
    t0 = time.time()
    out = pipe2.embed(x, batch_size=batch_size)
    emb = out[0] if isinstance(out, tuple) else out
    if isinstance(emb, list):
        pooled = np.stack([pool_embedding(e) for e in emb])
    else:
        e = emb.detach().float().cpu().numpy()
        pooled = e.reshape(e.shape[0], -1, e.shape[-1]).mean(axis=1)
    print(f"  [{desc}] {len(windows)} windows in {time.time()-t0:.1f}s")
    return pooled.astype(np.float32)

# --- Extract & SAVE ctx=100 embeddings (train-full + test-last) ---
EMB_TRAIN_PATH = os.path.join(RESULTS_DIR, "fd001_chronos_emb_train_ctx100.npy")
EMB_TEST_PATH  = os.path.join(RESULTS_DIR, "fd001_chronos_emb_test_ctx100.npy")

if os.path.exists(EMB_TRAIN_PATH) and os.path.exists(EMB_TEST_PATH):
    E_train = np.load(EMB_TRAIN_PATH)
    E_test  = np.load(EMB_TEST_PATH)
    print(f"✓ Embeddings loaded from disk: {E_train.shape}, {E_test.shape}")
else:
    Xw_train = build_windows = build_train_sequences(train, 100, "RUL_piecewise125")[0]
    E_train = extract_embeddings(Xw_train, desc="train-ctx100")
    Xw_test = build_last_window_sequences(test_full, 100, units_test)
    E_test  = extract_embeddings(Xw_test, desc="test-ctx100")
    np.save(EMB_TRAIN_PATH, E_train)
    np.save(EMB_TEST_PATH, E_test)
    print(f"✓ Embeddings extracted and SAVED to results/ "
          f"({E_train.shape}, {E_test.shape})")

# Labels aligned with embedding row order (unit asc, cycle asc)
y_train_pw = train.sort_values(["unit", "cycle"])["RUL_piecewise125"].values.astype(np.float32)
assert len(y_train_pw) == len(E_train)
print("\n✓ Section 1 complete: all machinery rebuilt, embeddings cached.")

In [ ]:
# ============================================================
# 08_Robustness.ipynb
# Section 2 — Multi-Seed Analysis (seeds 42-46, piecewise label)
# Purpose: Quantify run-to-run variance of the two GPU models.
#          LR/XGBoost excluded: deterministic (documented).
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEEDS = [42, 43, 44, 45, 46]

# --- 2a. LSTM: final config w=50, 128->64 (identical to NB05) ---
def build_lstm(window, units1, units2, lr=0.001, dropout=0.2):
    model = keras.Sequential([
        layers.Input(shape=(window, N_FEATURES)),
        layers.LSTM(units1, return_sequences=True),
        layers.Dropout(dropout),
        layers.LSTM(units2),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="mse")
    return model

print("Building LSTM sequences (w=50)...")
X_tr_lstm, y_tr_lstm = build_train_sequences(train, 50, "RUL_piecewise125")
X_te_lstm = build_last_window_sequences(test_full, 50, units_test)
Y_MAX_LSTM = float(train["RUL_piecewise125"].max())   # 125
print(f"  {X_tr_lstm.shape} -> test {X_te_lstm.shape}\n")

lstm_runs = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    model = build_lstm(50, 128, 64)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    model.fit(X_tr_lstm, y_tr_lstm / Y_MAX_LSTM, validation_split=0.1,
              epochs=50, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = model.predict(X_te_lstm, verbose=0).ravel() * Y_MAX_LSTM
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    lstm_runs.append({"seed": seed, "mae": mae, "rmse": rmse,
                      "nasa": ns, "train_s": t_train})
    print(f"  LSTM seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | "
          f"NASA {ns:8.2f} | {t_train:.0f}s")

# --- 2b. Chronos head: ctx=100, (128, 64) (identical to NB06) ---
def build_head(input_dim, arch=(128, 64)):
    L = [layers.Input(shape=(input_dim,))]
    for n in arch:
        L += [layers.Dense(n, activation="relu"), layers.Dropout(0.2)]
    L += [layers.Dense(1)]
    m = keras.Sequential(L)
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss="mse")
    return m

print("\n")
chronos_runs = []
for seed in SEEDS:
    keras.utils.set_random_seed(seed)
    head = build_head(E_train.shape[1])
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                       restore_best_weights=True)
    t0 = time.time()
    head.fit(E_train, y_train_pw / Y_MAX_LSTM, validation_split=0.1,
             epochs=100, batch_size=64, callbacks=[es], verbose=0)
    t_train = time.time() - t0
    y_pred = head.predict(E_test, verbose=0).ravel() * Y_MAX_LSTM
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    ns   = nasa_score(y_true, y_pred)
    chronos_runs.append({"seed": seed, "mae": mae, "rmse": rmse,
                        "nasa": ns, "train_s": t_train})
    print(f"  Chronos head seed {seed}: MAE {mae:6.2f} | RMSE {rmse:6.2f} | "
          f"NASA {ns:8.2f} | {t_train:.0f}s")

# --- 2c. Summary: mean ± std ---
def summarize(runs, name):
    df = pd.DataFrame(runs)
    print(f"\n{name} across seeds {SEEDS}:")
    for m in ["mae", "rmse", "nasa"]:
        print(f"  {m.upper():>5}: {df[m].mean():8.2f} ± {df[m].std():5.2f} "
              f"(min {df[m].min():.2f}, max {df[m].max():.2f})")
    return df

df_lstm = summarize(lstm_runs, "LSTM (w=50, 128→64, piecewise)")
df_chr  = summarize(chronos_runs, "Chronos-2 head (ctx=100, (128,64), piecewise)")

# --- 2d. The decisive check: does the ladder ordering survive? ---
print("\n=== Ladder robustness check (piecewise MAE) ===")
print(f"  XGBoost (deterministic, recorded): 13.21")
print(f"  LSTM worst seed:    {df_lstm['mae'].max():.2f} "
      f"{'< 13.21 ✓ gap robust' if df_lstm['mae'].max() < 13.21 else '>= 13.21 ⚠ overlaps XGBoost!'}")
print(f"  Chronos worst seed: {df_chr['mae'].max():.2f}")
print(f"  LSTM-vs-Chronos: LSTM worst {df_lstm['mae'].max():.2f} vs "
      f"Chronos best {df_chr['mae'].min():.2f} "
      f"{'-> no overlap, ordering robust' if df_lstm['mae'].max() < df_chr['mae'].min() else '-> RANGES OVERLAP (report honestly)'}")

# --- 2e. Save ---
df_lstm["model"] = "LSTM"; df_chr["model"] = "Chronos-2 head"
seed_df = pd.concat([df_lstm, df_chr], ignore_index=True)
seed_path = os.path.join(RESULTS_DIR, "fd001_multiseed_results.csv")
seed_df.to_csv(seed_path, index=False)
print(f"\nSaved: {seed_path}")

In [ ]:
# ============================================================
# 08 — Section 3: Wilcoxon Signed-Rank Tests (FAIR version)
# Uses the FAIR per-engine predictions (full history for all):
#   LR(C), XGBoost(C), LSTM(full), Chronos(full)
# Purpose: are the per-engine error differences significant?
# ============================================================
from scipy.stats import wilcoxon
from itertools import combinations

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")

# FAIR prediction files (every model saw each engine's full history)
preds_files = {
    "LR":      ("fd001_lr_full_history.csv",      "pred_LR_full"),
    "XGBoost": ("fd001_xgb_full_history.csv",     "pred_XGB_full"),
    "LSTM":    ("fd001_lstm_full_history.csv",    "pred_LSTM_full"),
    "Chronos": ("fd001_chronos_full_history.csv", "pred_Chronos_full"),
}

abs_err = {}
for name, (fn, col) in preds_files.items():
    df = pd.read_csv(os.path.join(RESULTS_DIR, fn))
    abs_err[name] = np.abs(df[col].values - df["RUL_true"].values)

print("=== Wilcoxon signed-rank tests: paired absolute errors ===")
print("    (n=100 engines, FAIR: all models see full history)\n")
print(f"{'Pair':<22} {'median Δ|err|':>14} {'p-value':>10} {'significant (α=0.05)':>22}")
print("-" * 72)
rows = []
for a, b in combinations(["LR", "XGBoost", "LSTM", "Chronos"], 2):
    stat, p = wilcoxon(abs_err[a], abs_err[b])
    med_delta = np.median(abs_err[a] - abs_err[b])
    sig = "YES" if p < 0.05 else "no"
    print(f"{a+' vs '+b:<22} {med_delta:>+14.2f} {p:>10.4f} {sig:>22}")
    rows.append({"model_a": a, "model_b": b,
                 "median_delta_abs_err": round(med_delta, 2),
                 "p_value": round(p, 4), "significant_5pct": sig})

wilc_df = pd.DataFrame(rows)
wilc_path = os.path.join(RESULTS_DIR, "fd001_wilcoxon_tests_fair.csv")
wilc_df.to_csv(wilc_path, index=False)
print(f"\nSaved: {wilc_path}")

In [ ]:
# ============================================================
# 08_Robustness.ipynb
# Section 4 — Per-Engine RUL Trajectory Plots (piecewise)
# Purpose: Predict at EVERY cycle of 4 selected test engines
#          with all four models; plot predicted vs actual RUL.
#          Truth line is the piecewise-125 (capped) target.
# ============================================================
from sklearn.linear_model import LinearRegression
import xgboost as xgb

# --- 4a. Retrain the four final models (seed 42, piecewise) ---
X_train_flat = train[FINAL_FEATURES_FD001].values
y_train_flat = train["RUL_piecewise125"].values

lr_model = LinearRegression().fit(X_train_flat, y_train_flat)

xgb_model = xgb.XGBRegressor(max_depth=8, n_estimators=300, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=42, n_jobs=-1,
                             objective="reg:squarederror")
xgb_model.fit(X_train_flat, y_train_flat)
print("✓ LR + XGBoost retrained (deterministic).")

keras.utils.set_random_seed(42)
lstm_model = build_lstm(50, 128, 64)
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                   restore_best_weights=True)
lstm_model.fit(X_tr_lstm, y_tr_lstm / Y_MAX_LSTM, validation_split=0.1,
               epochs=50, batch_size=64, callbacks=[es], verbose=0)

keras.utils.set_random_seed(42)
head_model = build_head(E_train.shape[1])
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                   restore_best_weights=True)
head_model.fit(E_train, y_train_pw / Y_MAX_LSTM, validation_split=0.1,
               epochs=100, batch_size=64, callbacks=[es], verbose=0)
print("✓ LSTM + Chronos head retrained (seed 42).")

# --- 4b. Select 4 engines with printed rationale ---
hist_len = test_full.groupby("unit")["cycle"].max()
sel = {
    "longest history":  int(hist_len.idxmax()),
    "shortest history": int(hist_len.idxmin()),
    "lowest true RUL":  int(units_test[np.argmin(y_true)]),
    "highest true RUL": int(units_test[np.argmax(y_true)]),
}
seen, engines = set(), []
for reason, u in sel.items():
    if u not in seen:
        engines.append((u, reason)); seen.add(u)
print("\nSelected engines:")
for u, reason in engines:
    ru = y_true[units_test == u][0]
    print(f"  engine {u:>3}: {reason:<17} | history {hist_len[u]:>3} cycles "
          f"| true RUL at last cycle {ru:.0f}")

# --- 4c. Per-cycle predictions for each selected engine ---
def engine_trajectory_predictions(unit):
    grp = test_full[test_full["unit"] == unit].sort_values("cycle")
    feats = grp[FINAL_FEATURES_FD001].values
    cycles = grp["cycle"].values
    L = len(grp)
    rul_last = float(y_true[units_test == unit][0])
    true_rul = np.minimum(rul_last + (cycles[-1] - cycles), 125.0)  # piecewise-125 truth
    p_lr  = lr_model.predict(feats)
    p_xgb = xgb_model.predict(feats)
    Xw50 = np.stack([np.vstack([np.zeros((max(0, 50-(i+1)), N_FEATURES)),
                                feats[max(0, i-49):i+1]])
                     for i in range(L)]).astype(np.float32)
    p_lstm = lstm_model.predict(Xw50, verbose=0).ravel() * Y_MAX_LSTM
    Xw100 = np.stack([np.vstack([np.zeros((max(0, 100-(i+1)), N_FEATURES)),
                                 feats[max(0, i-99):i+1]])
                      for i in range(L)]).astype(np.float32)
    E = extract_embeddings(Xw100, desc=f"engine{unit}")
    p_chr = head_model.predict(E, verbose=0).ravel() * Y_MAX_LSTM
    return cycles, true_rul, {"Linear Regression": p_lr, "XGBoost": p_xgb,
                              "LSTM": p_lstm, "Chronos-2 + head": p_chr}

MODEL_COLORS = {"Linear Regression": "#999999", "XGBoost": "#2E75B6",
                "LSTM": "#C00000", "Chronos-2 + head": "#548235"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (unit, reason) in zip(axes.ravel(), engines):
    cycles, true_rul, preds_traj = engine_trajectory_predictions(unit)
    ax.plot(cycles, true_rul, "k--", linewidth=1.8, label="Actual RUL")
    ax.axhline(125, color="black", linewidth=0.6, alpha=0.4)
    for name, p in preds_traj.items():
        ax.plot(cycles, p, linewidth=1.3, color=MODEL_COLORS[name],
                label=name, alpha=0.9)
    ax.set_xlabel("Cycle")
    ax.set_ylabel("RUL (cycles)")
    rul_phys = float(y_true[units_test == unit][0])
    ax.set_title(f"Test engine {unit} — {reason}\n"
                 f"(history {cycles[-1]:.0f} cycles, "
                 f"true RUL at last cycle {rul_phys:.0f}"
                 + (", shown capped at 125" if rul_phys > 125 else "") + ")")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True, alpha=0.25)

fig.suptitle("FD001 — Predicted vs. actual RUL over the observed life of four "
             "selected test engines\n(piecewise-125 label; horizontal line = "
             "125-cycle cap)", fontsize=13, y=1.00)
fig.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "fd001_rul_trajectories.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {fig_path}")

In [ ]:
# ============================================================
# 08_Robustness.ipynb
# Section 5 — Save Trajectory Data and Finalize
# ============================================================
rows = []
for unit, reason in engines:
    cycles, true_rul, preds_traj = engine_trajectory_predictions(unit)
    for i, c in enumerate(cycles):
        rows.append({"unit": unit, "reason": reason, "cycle": int(c),
                     "true_RUL_piecewise125": float(true_rul[i]),
                     **{k.replace(" ", "_").replace("-", "_").replace("+", "plus"):
                        float(v[i]) for k, v in preds_traj.items()}})
traj_df = pd.DataFrame(rows)
traj_path = os.path.join(RESULTS_DIR, "fd001_rul_trajectories.csv")
traj_df.to_csv(traj_path, index=False)
print(f"Saved: {traj_path}  ({len(traj_df)} rows, capped truth)")

print("\n" + "=" * 60)
print("NOTEBOOK 08 COMPLETE — ROBUSTNESS PHASE FINISHED")
print("=" * 60)
print("""
Artifacts:
  results/fd001_multiseed_results.csv    — 5 seeds x 2 GPU models
  results/fd001_wilcoxon_tests.csv       — 6 pairwise tests
  results/fd001_rul_trajectories.csv     — per-cycle predictions, 4 engines
  results/fd001_chronos_emb_train_ctx100.npy — cached embeddings (train)
  results/fd001_chronos_emb_test_ctx100.npy  — cached embeddings (test)
  figures/fd001_rul_trajectories.png     — the classic RUL figure
""")
print("=" * 60)